# Water use analysis

In [ ]:
# basic
from pathlib import Path
import pandas as pd
import yaml
import os

# spatial
import geopandas as gpd
import xarray as xr

path_data_raw = Path("/home/rooda/Datasets")
wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes')
os.chdir(wd)

# import config 
with open(wd / "config.yml") as f:
    cfg = yaml.safe_load(f)

from src.utils import extract_timeseries

## Data

In [ ]:
common_period = slice("2000-01-01", "2010-12-31") # common period for all datasets

# metadata (basins)
metadata_hydroatlas = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg")
metadata_hydroatlas = metadata_hydroatlas[metadata_hydroatlas.glacier_area_RGI60 > 0] # only the modeled ones

In [ ]:
# water use data from Huang et al. (only consumptive use in mm/month)
ds_cons_total = xr.open_dataset(path_data_raw / "WATER_USE/Huang_HESS/Water_use_Huang.nc").cons_total
ds_cons_total = ds_cons_total.sel(date=common_period)
ds_cons_total = extract_timeseries(ds_cons_total, metadata_hydroatlas)

# to daily and mean daily cycle
ds_cons_total = ds_cons_total.resample("D").ffill() 
ds_cons_total = ds_cons_total.div(ds_cons_total.index.days_in_month, axis=0)
ds_cons_total = ds_cons_total.groupby(ds_cons_total.index.dayofyear).mean()
#ds_cons_total = (ds_cons_total * metadata_hydroatlas.set_index("gauge_id")["basin_area"]*1e6) / (1000*86400) # to m3/s

In [ ]:
# water availability (near-natural water availability + what if scenarios)
runoff_hydroatlas_hist = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_hist.nc").q_mm_sim
runoff_hydroatlas_hist = runoff_hydroatlas_hist.expand_dims(whatif_scenario=['historical'])
runoff_hydroatlas_whatif = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_whatif.nc").q_mm_sim
runoff_hydroatlas_whatif = runoff_hydroatlas_whatif.groupby('whatif_scenario').mean() # multi model mean

# merge both datasets
runoff_hydroatlas_all = xr.concat([runoff_hydroatlas_hist, runoff_hydroatlas_whatif], dim='whatif_scenario')
runoff_hydroatlas_all = runoff_hydroatlas_all.sel(date=common_period)
runoff_hydroatlas_all = runoff_hydroatlas_all.rolling(date=30, center=True).mean() # 30-day rolling mean to smooth the data
runoff_hydroatlas_all = runoff_hydroatlas_all.groupby(runoff_hydroatlas_all.date.dt.dayofyear).mean()
#runoff_hydroatlas_all = runoff_hydroatlas_all *(metadata_hydroatlas.basin_area.values*1e6)[:, None] / (1000*86400)
runoff_hydroatlas_all = runoff_hydroatlas_all.to_series().unstack(["whatif_scenario", "gauge_id"])

## WSI

In [ ]:
# water stress index (WSI) = consumption / runoff
wsi_scenarios = {scenario: ds_cons_total / runoff_hydroatlas_all[scenario] 
                 for scenario in runoff_hydroatlas_all.columns.get_level_values("whatif_scenario").unique()}
wsi_scenarios = pd.concat(wsi_scenarios, axis=0)
wsi_scenarios = pd.concat([wsi_scenarios, pd.concat({"water demand": ds_cons_total}, axis=0)], axis=0)
wsi_scenarios = pd.concat([wsi_scenarios, pd.concat({"runoff": runoff_hydroatlas_all["historical"]}, axis=0)], axis=0)

# reformat and save
wsi_scenarios = wsi_scenarios.melt(ignore_index=False, value_name="WSI", var_name="gauge_id")
wsi_scenarios = wsi_scenarios.reset_index().rename(columns={"level_0": "scenario", "level_1": "dayofyear"})
wsi_scenarios.to_parquet("data/hydro/water_stress.parquet")